In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in (
            NOTEBOOK_WORKING_DIRECTORY,
            *NOTEBOOK_WORKING_DIRECTORY.parents,
        )
        if (
            (candidate_directory / "reports" / "tables").is_dir()
            and (candidate_directory / "models" / "metrics").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root containing reports/tables "
        "and models/metrics."
    )

os.chdir(PROJECT_ROOT)

TABLE_DIRECTORY = PROJECT_ROOT / "reports" / "tables"
FIGURE_DIRECTORY = PROJECT_ROOT / "reports" / "figures"
REPORT_DIRECTORY = PROJECT_ROOT / "reports" / "evaluation"

FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
REPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

POLICY_ERROR_SUMMARY_PATH = (
    TABLE_DIRECTORY / "phase7b_policy_error_summary.csv"
)

CAPACITY_SENSITIVITY_PATH = (
    TABLE_DIRECTORY / "phase7c_capacity_sensitivity.csv"
)

VALIDATION_HOLDOUT_COMPARISON_PATH = (
    TABLE_DIRECTORY / "phase7c_validation_holdout_policy_comparison.csv"
)

VALIDATION_POLICY_PATH = (
    PROJECT_ROOT
    / "models"
    / "metrics"
    / "phase5_validation_policy_evaluation.json"
)

policy_error_summary = pd.read_csv(POLICY_ERROR_SUMMARY_PATH)
capacity_sensitivity = pd.read_csv(CAPACITY_SENSITIVITY_PATH)
validation_holdout_comparison = pd.read_csv(
    VALIDATION_HOLDOUT_COMPARISON_PATH
)

with VALIDATION_POLICY_PATH.open(encoding="utf-8") as file:
    validation_policy_source = json.load(file)

if len(policy_error_summary) != 1:
    raise ValueError(
        "Expected exactly one Phase 7B final-holdout policy summary row."
    )

policy = policy_error_summary.iloc[0].to_dict()

cost_assumptions = validation_policy_source["cost_assumptions"]

false_negative_cost = float(
    cost_assumptions["false_negative_cost"]
)

manual_review_cost = float(
    cost_assumptions["manual_review_cost"]
)

fraud_prevention_value = float(
    cost_assumptions["fraud_prevention_value"]
)

currency = "GBP"

captured_fraud_count = int(policy["captured_fraud_count"])
missed_fraud_count = int(policy["missed_fraud_count"])
unnecessary_review_count = int(policy["unnecessary_review_count"])
reviewed_transaction_count = int(policy["reviewed_transaction_count"])
total_fraud_count = int(policy["total_fraud_count"])
review_capacity = int(policy["review_capacity"])

total_expected_review_cost = (
    reviewed_transaction_count * manual_review_cost
)

total_expected_prevention_value = (
    captured_fraud_count * fraud_prevention_value
)

net_expected_value = (
    total_expected_prevention_value
    - total_expected_review_cost
)

illustrative_residual_missed_fraud_exposure = (
    missed_fraud_count * false_negative_cost
)

review_precision = captured_fraud_count / reviewed_transaction_count
fraud_capture_rate = captured_fraud_count / total_fraud_count

business_value_summary = pd.DataFrame(
    [
        {
            "model_name": policy["model_name"],
            "model_version": policy["model_version"],
            "mlflow_run_id": policy["mlflow_run_id"],
            "evaluation_period": "final_chronological_holdout",
            "currency": currency,
            "review_capacity": review_capacity,
            "reviewed_transaction_count": reviewed_transaction_count,
            "total_fraud_count": total_fraud_count,
            "captured_fraud_count": captured_fraud_count,
            "missed_fraud_count": missed_fraud_count,
            "unnecessary_review_count": unnecessary_review_count,
            "fraud_capture_rate": fraud_capture_rate,
            "review_precision": review_precision,
            "manual_review_cost_per_case": manual_review_cost,
            "fraud_prevention_value_per_captured_case": (
                fraud_prevention_value
            ),
            "false_negative_cost_per_missed_case": false_negative_cost,
            "total_expected_review_cost": total_expected_review_cost,
            "total_expected_prevention_value": (
                total_expected_prevention_value
            ),
            "net_expected_value": net_expected_value,
            "illustrative_residual_missed_fraud_exposure": (
                illustrative_residual_missed_fraud_exposure
            ),
            "interpretation": (
                "Public-data benchmark estimate under documented "
                "illustrative assumptions; not realised institutional value."
            ),
        }
    ]
)

business_value_summary.to_csv(
    TABLE_DIRECTORY / "phase7d_business_value_summary.csv",
    index=False,
)

capacity_business_value = capacity_sensitivity.copy()

capacity_business_value["review_cost_per_case"] = manual_review_cost
capacity_business_value["fraud_prevention_value_per_case"] = (
    fraud_prevention_value
)
capacity_business_value["false_negative_cost_per_case"] = (
    false_negative_cost
)
capacity_business_value["estimated_missed_fraud_count"] = (
    capacity_business_value["total_fraud_count"]
    - capacity_business_value["captured_fraud_count"]
)
capacity_business_value["illustrative_residual_missed_fraud_exposure"] = (
    capacity_business_value["estimated_missed_fraud_count"]
    * false_negative_cost
)

capacity_business_value.to_csv(
    TABLE_DIRECTORY / "phase7d_capacity_business_value.csv",
    index=False,
)

policy_value_components = pd.DataFrame(
    [
        {
            "component": "Expected prevention value from captured fraud",
            "amount_gbp": total_expected_prevention_value,
        },
        {
            "component": "Manual review cost",
            "amount_gbp": -total_expected_review_cost,
        },
        {
            "component": "Net expected value",
            "amount_gbp": net_expected_value,
        },
        {
            "component": "Residual missed-fraud exposure",
            "amount_gbp": -illustrative_residual_missed_fraud_exposure,
        },
    ]
)

policy_value_components.to_csv(
    TABLE_DIRECTORY / "phase7d_policy_value_components.csv",
    index=False,
)

validation_holdout_value_comparison = (
    validation_holdout_comparison.copy()
)

validation_holdout_value_comparison["review_precision"] = (
    validation_holdout_value_comparison["captured_fraud_count"]
    / validation_holdout_value_comparison["selected_review_count"]
)

validation_holdout_value_comparison.to_csv(
    TABLE_DIRECTORY / "phase7d_validation_holdout_value_comparison.csv",
    index=False,
)

plt.figure(figsize=(11, 6))

value_component_colours = [
    "#16A34A",
    "#DC2626",
    "#2563EB",
    "#F59E0B",
]

plt.bar(
    policy_value_components["component"],
    policy_value_components["amount_gbp"],
    color=value_component_colours,
)

plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=20, ha="right")
plt.ylabel("Illustrative value (GBP)")
plt.title(
    "Phase 7D: Final Holdout Policy Value Components"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7d_holdout_value_components.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

plt.figure(figsize=(11, 6))

plt.plot(
    capacity_business_value["review_capacity"],
    capacity_business_value["net_expected_value"],
    marker="o",
    linewidth=2,
    color="#2563EB",
    label="Illustrative net expected value",
)

plt.plot(
    capacity_business_value["review_capacity"],
    capacity_business_value[
        "illustrative_residual_missed_fraud_exposure"
    ],
    marker="o",
    linewidth=2,
    color="#DC2626",
    label="Illustrative residual missed-fraud exposure",
)

plt.xlabel("Review capacity")
plt.ylabel("Illustrative value / exposure (GBP)")
plt.title(
    "Phase 7D: Validation Capacity Trade-off"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7d_capacity_value_tradeoff.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()


def markdown_table(
    table: pd.DataFrame,
    columns: list[str],
    decimal_columns: list[str] | None = None,
) -> str:
    """Create a Markdown table without optional dependencies."""
    decimal_columns = decimal_columns or []
    display_table = table.loc[:, columns].copy()

    for column in decimal_columns:
        if column in display_table.columns:
            display_table[column] = display_table[column].map(
                lambda value: (
                    f"{float(value):,.6f}"
                    if pd.notna(value)
                    else ""
                )
            )

    display_table = display_table.fillna("")

    for column in display_table.columns:
        display_table[column] = (
            display_table[column]
            .astype(str)
            .str.replace("|", "\\|", regex=False)
        )

    header = "| " + " | ".join(display_table.columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(display_table.columns)
    ) + " |"

    rows = [
        "| " + " | ".join(row) + " |"
        for row in display_table.astype(str).values.tolist()
    ]

    return "\n".join([header, separator, *rows])


capacity_500 = capacity_business_value.loc[
    capacity_business_value["review_capacity"] == 500
].iloc[0]

capacity_1000 = capacity_business_value.loc[
    capacity_business_value["review_capacity"] == 1000
].iloc[0]

capacity_2000 = capacity_business_value.loc[
    capacity_business_value["review_capacity"] == 2000
].iloc[0]

report_content = f"""# Phase 7D — Business Value Report

## Purpose

This report translates the frozen fraud-prioritisation model's final-holdout
performance into review volume, fraud capture, missed fraud, illustrative review
cost, illustrative prevention value, and constrained-capacity trade-offs.

All monetary figures are benchmark calculations under documented illustrative
assumptions. They are not realised savings, actual fraud losses, institutional
forecasts, or financial advice.

## Frozen Evaluation Scope

| Item | Value |
| --- | --- |
| Model name | `{policy["model_name"]}` |
| Model version | `{policy["model_version"]}` |
| MLflow training run | `{policy["mlflow_run_id"]}` |
| Evaluation period | Final chronological holdout |
| Review capacity | `{review_capacity:,}` |
| Currency | `{currency}` |
| Manual review cost per reviewed transaction | `{currency} {manual_review_cost:,.2f}` |
| Fraud prevention value per captured fraud case | `{currency} {fraud_prevention_value:,.2f}` |
| False-negative cost per missed fraud case | `{currency} {false_negative_cost:,.2f}` |

The model, calibration method, and capacity policy were selected using earlier
chronological validation evidence. The final holdout is reported here as the
locked benchmark evaluation, not used for additional model or policy selection.

## Final Holdout Outcome

At the locked review capacity of `{review_capacity:,}`, the policy reviewed
`{reviewed_transaction_count:,}` transactions and captured
`{captured_fraud_count:,}` of `{total_fraud_count:,}` fraud-labelled transactions.
This corresponds to a fraud capture rate of `{fraud_capture_rate:.6f}` and review
precision of `{review_precision:.6f}`.

- Captured fraud: `{captured_fraud_count:,}`
- Missed fraud: `{missed_fraud_count:,}`
- Legitimate transactions reviewed: `{unnecessary_review_count:,}`
- Total review volume: `{reviewed_transaction_count:,}`

{markdown_table(
    business_value_summary,
    columns=[
        "review_capacity",
        "reviewed_transaction_count",
        "total_fraud_count",
        "captured_fraud_count",
        "missed_fraud_count",
        "unnecessary_review_count",
        "fraud_capture_rate",
        "review_precision",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
        "illustrative_residual_missed_fraud_exposure",
    ],
    decimal_columns=[
        "fraud_capture_rate",
        "review_precision",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
        "illustrative_residual_missed_fraud_exposure",
    ],
)}

## Illustrative Value Components

![Final holdout value components](../figures/phase7d_holdout_value_components.png)

Under the documented assumptions, the `{reviewed_transaction_count:,}` reviews
have an illustrative manual review cost of `{currency} {total_expected_review_cost:,.2f}`.
The `{captured_fraud_count:,}` captured fraud cases correspond to illustrative
prevention value of `{currency} {total_expected_prevention_value:,.2f}`, producing
illustrative net expected value of `{currency} {net_expected_value:,.2f}`.

The `{missed_fraud_count:,}` missed fraud-labelled cases correspond to an
illustrative residual missed-fraud exposure of
`{currency} {illustrative_residual_missed_fraud_exposure:,.2f}` if every missed
case incurs the documented false-negative cost. This figure is shown separately:
it is an exposure indicator, not an amount subtracted from the saved Phase 5 net
expected value calculation.

## Capacity Trade-offs

![Capacity value trade-off](../figures/phase7d_capacity_value_tradeoff.png)

{markdown_table(
    capacity_business_value,
    columns=[
        "review_capacity",
        "selected_review_count",
        "captured_fraud_count",
        "total_fraud_count",
        "fraud_capture_rate",
        "false_positive_review_count",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
        "illustrative_residual_missed_fraud_exposure",
    ],
    decimal_columns=[
        "fraud_capture_rate",
        "total_expected_review_cost",
        "total_expected_prevention_value",
        "net_expected_value",
        "illustrative_residual_missed_fraud_exposure",
    ],
)}

Increasing validation review capacity from `{int(capacity_500["review_capacity"]):,}`
to `{int(capacity_2000["review_capacity"]):,}` increased fraud capture rate from
`{float(capacity_500["fraud_capture_rate"]):.6f}` to
`{float(capacity_2000["fraud_capture_rate"]):.6f}`. It also increased review
volume, manual review costs, and legitimate-review burden.

The configured 1,000-case capacity is a documented benchmark assumption. It
illustrates the operational trade-off between review capacity, fraud capture, and
review burden; it is not a recommendation for any real financial institution.

## Validation Versus Holdout

{markdown_table(
    validation_holdout_value_comparison,
    columns=[
        "evaluation_period",
        "review_capacity",
        "selected_review_count",
        "captured_fraud_count",
        "total_fraud_count",
        "fraud_capture_rate",
        "review_precision",
        "false_positive_review_count",
        "net_expected_value",
    ],
    decimal_columns=[
        "fraud_capture_rate",
        "review_precision",
        "net_expected_value",
    ],
)}

The validation and holdout periods contain different transaction and fraud volumes.
Therefore, absolute illustrative net expected value should not be compared as
though the periods were identical. More comparable indicators are capacity,
review volume, fraud capture rate, review precision, and the documented cost
assumptions.

## Limitations

- The IEEE-CIS dataset is a public benchmark with obfuscated variables and does
  not represent a live financial-institution operating environment.
- Review cost, prevention value, and false-negative cost are illustrative
  assumptions documented for portfolio decision modelling.
- Captured fraud labels are benchmark outcomes; they do not represent prevented
  loss or confirmed operational intervention.
- The residual missed-fraud exposure estimate assumes the same false-negative cost
  for every missed fraud case and should not be interpreted as actual loss.
- The selected policy reflects a fixed capacity constraint. A different review
  capacity, investigation quality, customer-friction cost, or intervention
  effectiveness would change the decision value.
- These figures are not causal estimates and must not be presented as realised
  savings or institutional performance.

## Output Inventory

```text
reports/figures/phase7d_holdout_value_components.png
reports/figures/phase7d_capacity_value_tradeoff.png
reports/tables/phase7d_business_value_summary.csv
reports/tables/phase7d_capacity_business_value.csv
reports/tables/phase7d_policy_value_components.csv
reports/tables/phase7d_validation_holdout_value_comparison.csv
reports/evaluation/business_value_report.md
```
"""

report_path = REPORT_DIRECTORY / "business_value_report.md"
report_path.write_text(report_content, encoding="utf-8")

print("=== PHASE 7D BUSINESS VALUE COMPLETE ===")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Report size: {report_path.stat().st_size:,} bytes")
print(
    "Final holdout illustrative net expected value: "
    f"{currency} {net_expected_value:,.2f}"
)
print(
    "Final holdout illustrative residual missed-fraud exposure: "
    f"{currency} {illustrative_residual_missed_fraud_exposure:,.2f}"
)
print(
    "Capacity range evaluated: "
    f"{int(capacity_business_value['review_capacity'].min()):,} to "
    f"{int(capacity_business_value['review_capacity'].max()):,}"
)